# Bloco 3 — Notebook 04: Backtest Científico das Camadas 1, 2 e 3 do Radar
Valida a contribuição incremental das camadas históricas do **QuantCrypto Radar & Analyst Engine (v1.1)** sob a mesma governança anti-overfitting (partição 60/25/15, AFML sample uniqueness e registro em `logs/backtest_trials_registry.csv`):
- **Camada 1 (Estrutura EMA 9/50/200 + Clima MECE 4h)**
- **Camada 2 (Price Action: Pinbar, Engolfo de Alta e Liquidity Sweep)**
- **Camada 3 (Order Flow: RVOL $\ge 1,5$ e Taker Buy Ratio $\ge 0,55$)**


In [ ]:
# Se estiver rodando no Google Colab, clona o repositório e instala dependências leves
import os, sys
from pathlib import Path
if 'google.colab' in sys.modules and not Path('config/strategies.yaml').exists():
    !git clone https://github.com/pac-coins/quant-crypto-chameleon-radar-engine.git
    os.chdir('quant-crypto-chameleon-radar-engine')
    !pip install -q pydantic pydantic-settings PyYAML pandas requests cryptography
if '.' not in sys.path:
    sys.path.insert(0, '.')


In [ ]:
import pandas as pd
from src.core.config import load_config
from src.features.feature_store import FeatureStore, FeatureRequest
from src.regimes.regime_detector import MarketRegime

cfg = load_config()

def score_radar_layers_123(row: pd.Series, regime: MarketRegime) -> int:
    score = 0
    ema9 = row.get('ema_9', row.get('ema_9_1h', 0.0))
    ema50 = row.get('ema_50', row.get('ema_50_1h', 0.0))
    ema200 = row.get('ema_200', row.get('ema_200_1h', 0.0))
    if regime == MarketRegime.BULL and ema9 > ema50 > ema200:
        score += 30
    if row.get('liquidity_sweep_bull_20', row.get('is_liquidity_sweep', False)):
        score += 25
    elif row.get('is_bullish_pinbar', False) or row.get('is_bullish_engulfing', False):
        score += 20
    rvol = row.get('rvol_20', row.get('rvol_20_1h', 0.0))
    tbr = row.get('tbr', row.get('taker_buy_ratio', 0.5))
    if rvol >= 1.5 and tbr >= 0.55:
        score += 20
    return score
